In [ ]:
# Cell 1 - Clone & Install
!git clone https://github.com/amrmalkhatib/Emotional-Tone.git
!pip install numpy==1.24.3 gensim==4.3.1 scipy==1.10.1 -q

In [ ]:
# Cell 2 - Imports
import pandas as pd
import numpy as np
import re, time
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
print('Done!')

In [ ]:
# Cell 3 - Load & Preprocess
df = pd.read_csv('Emotional-Tone/Emotional-Tone-Dataset.csv')
df.columns = df.columns.str.strip()
arabic_sw = set(stopwords.words('arabic'))

def clean(text):
    if pd.isna(text): return ''
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return ' '.join([w for w in text.split() if w not in arabic_sw])

df['clean_text'] = df['TWEET'].apply(clean)
y = df['LABEL']
print('Shape:', df.shape)

In [ ]:
# Cell 4 - Representations
X_bow   = CountVectorizer(max_features=5000).fit_transform(df['clean_text']).toarray()
X_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2)).fit_transform(df['clean_text']).toarray()

sents = df['clean_text'].apply(str.split).tolist()
w2v = Word2Vec(sents, vector_size=100, window=5, min_count=1, workers=4, epochs=10)
X_w2v = np.array([np.mean([w2v.wv[w] for w in t if w in w2v.wv] or [np.zeros(100)], axis=0) for t in sents])

print('BoW:', X_bow.shape, '| TF-IDF:', X_tfidf.shape, '| W2V:', X_w2v.shape)

In [ ]:
# Cell 5 - Classical ML Search (3 configs per model)
param_grid = {
    'Naive Bayes': [
        {'model': MultinomialNB(alpha=0.5),  'p': 'alpha=0.5'},
        {'model': MultinomialNB(alpha=1.0),  'p': 'alpha=1.0'},
        {'model': MultinomialNB(alpha=2.0),  'p': 'alpha=2.0'},
    ],
    'Decision Tree': [
        {'model': DecisionTreeClassifier(max_depth=5,  random_state=42), 'p': 'depth=5'},
        {'model': DecisionTreeClassifier(max_depth=10, random_state=42), 'p': 'depth=10'},
        {'model': DecisionTreeClassifier(max_depth=20, random_state=42), 'p': 'depth=20'},
    ],
    'Random Forest': [
        {'model': RandomForestClassifier(n_estimators=50,  max_depth=10, random_state=42), 'p': 'n=50,d=10'},
        {'model': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42), 'p': 'n=100,d=10'},
        {'model': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42), 'p': 'n=100,d=15'},
    ],
    'AdaBoost': [
        {'model': AdaBoostClassifier(n_estimators=50,  learning_rate=0.5, random_state=42), 'p': 'n=50,lr=0.5'},
        {'model': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42), 'p': 'n=100,lr=0.5'},
        {'model': AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42), 'p': 'n=100,lr=1.0'},
    ],
    'SVM': [
        {'model': SVC(kernel='rbf',    C=1,  gamma='scale'), 'p': 'rbf,C=1'},
        {'model': SVC(kernel='rbf',    C=10, gamma='scale'), 'p': 'rbf,C=10'},
        {'model': SVC(kernel='linear', C=1               ), 'p': 'linear,C=1'},
    ],
}

reps = {'BoW': X_bow, 'TF-IDF': X_tfidf, 'Word2Vec': X_w2v}
w2v_skip = ['Naive Bayes']  # MultinomialNB needs non-negative
all_results = []
best = {'f1': 0}

for rep_name, X in reps.items():
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    for mname, cfgs in param_grid.items():
        if rep_name == 'Word2Vec' and mname in w2v_skip:
            continue
        for cfg in cfgs:
            try:
                cfg['model'].fit(Xtr, ytr)
                f1 = f1_score(yte, cfg['model'].predict(Xte), average='macro', zero_division=0)
                all_results.append({'Rep': rep_name, 'Model': mname, 'Params': cfg['p'], 'F1': round(f1,4)})
                if f1 > best['f1']:
                    best = {'f1': f1, 'model': mname, 'params': cfg['p'], 'rep': rep_name, 'obj': cfg['model']}
            except Exception as e:
                print(f'Skip {mname} {cfg["p"]} {rep_name}: {e}')

res_df = pd.DataFrame(all_results).sort_values('F1', ascending=False)
print('\n📊 Classical ML Results:')
print(res_df.to_string(index=False))
res_df.to_csv('classical_results.csv', index=False)
print(f'\n🏆 Best: {best["model"]} | {best["params"]} | {best["rep"]} | F1={best["f1"]:.4f}')

In [ ]:
# Cell 6 - Best Classical ML Full Report
Xb = {'BoW': X_bow, 'TF-IDF': X_tfidf, 'Word2Vec': X_w2v}[best['rep']]
Xtr, Xte, ytr, yte = train_test_split(Xb, y, test_size=0.2, random_state=42, stratify=y)
best['obj'].fit(Xtr, ytr)
print('🏆 Best Model Full Report:')
print(classification_report(yte, best['obj'].predict(Xte), zero_division=0))

In [ ]:
# Cell 7 - FNN Search (3 configs, 10 epochs)
le = LabelEncoder()
y_enc = le.fit_transform(df['LABEL'].values)
y_cat = to_categorical(y_enc)
Xf = TfidfVectorizer(max_features=5000).fit_transform(df['clean_text']).toarray()
Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(Xf, y_cat, test_size=0.2, random_state=42, stratify=y_enc)
yte_cls = np.argmax(yte_f, axis=1)

fnn_cfgs = [
    {'lr': 0.01,  'drop': 0.2, 'units': 128},
    {'lr': 0.001, 'drop': 0.3, 'units': 256},
    {'lr': 0.001, 'drop': 0.5, 'units': 256},
]
fnn_res = []
best_fnn = {'f1': 0}

for i, c in enumerate(fnn_cfgs):
    print(f'\nFNN {i+1}: lr={c["lr"]}, drop={c["drop"]}, units={c["units"]}')
    m = Sequential([
        Dense(c['units'], activation='relu', input_shape=(Xtr_f.shape[1],)),
        Dropout(c['drop']),
        Dense(c['units']//2, activation='relu'),
        Dropout(c['drop']),
        Dense(y_cat.shape[1], activation='softmax')
    ])
    m.compile(optimizer=Adam(c['lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(Xtr_f, ytr_f, validation_split=0.2, epochs=10, batch_size=32, verbose=0)
    yp = np.argmax(m.predict(Xte_f, verbose=0), axis=1)
    f1 = f1_score(yte_cls, yp, average='macro', zero_division=0)
    acc = (yp == yte_cls).mean()
    print(f'Acc={acc:.4f} | F1={f1:.4f}')
    fnn_res.append({**c, 'F1': round(f1,4), 'Acc': round(acc,4)})
    if f1 > best_fnn['f1']:
        best_fnn = {'f1': f1, 'cfg': c, 'model': m, 'pred': yp}

print('\n📊 FNN Results:')
print(pd.DataFrame(fnn_res).sort_values('F1', ascending=False).to_string(index=False))
print(f'\n🏆 Best FNN: lr={best_fnn["cfg"]["lr"]}, drop={best_fnn["cfg"]["drop"]}, units={best_fnn["cfg"]["units"]} | F1={best_fnn["f1"]:.4f}')
print(classification_report(yte_cls, best_fnn['pred'], target_names=le.classes_, zero_division=0))

In [ ]:
# Cell 8 - LSTM Search (3 configs, 10 epochs)
tok = Tokenizer(num_words=10000, oov_token='<OOV>')
tok.fit_on_texts(df['clean_text'])
Xpad = pad_sequences(tok.texts_to_sequences(df['clean_text']), maxlen=100, padding='post')
le2 = LabelEncoder()
yl = le2.fit_transform(df['LABEL'].values)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(Xpad, yl, test_size=0.2, random_state=42, stratify=yl)
nc = len(np.unique(yl))

lstm_cfgs = [
    {'units': 64,  'embed': 64,  'lr': 0.001},
    {'units': 64,  'embed': 128, 'lr': 0.001},
    {'units': 128, 'embed': 128, 'lr': 0.0005},
]
lstm_res = []
best_lstm = {'f1': 0}

for i, c in enumerate(lstm_cfgs):
    print(f'\nLSTM {i+1}: units={c["units"]}, embed={c["embed"]}, lr={c["lr"]}')
    m = Sequential([
        Embedding(10000, c['embed'], input_length=100),
        Bidirectional(LSTM(c['units'], return_sequences=True)),
        Bidirectional(LSTM(c['units']//2)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(nc, activation='softmax')
    ])
    m.compile(optimizer=Adam(c['lr']), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    m.fit(Xtr_l, ytr_l, validation_data=(Xte_l, yte_l), epochs=10, batch_size=32, verbose=0)
    yp = np.argmax(m.predict(Xte_l, verbose=0), axis=1)
    f1 = f1_score(yte_l, yp, average='macro', zero_division=0)
    acc = (yp == yte_l).mean()
    print(f'Acc={acc:.4f} | F1={f1:.4f}')
    lstm_res.append({**c, 'F1': round(f1,4), 'Acc': round(acc,4)})
    if f1 > best_lstm['f1']:
        best_lstm = {'f1': f1, 'cfg': c, 'model': m, 'pred': yp}

print('\n📊 LSTM Results:')
print(pd.DataFrame(lstm_res).sort_values('F1', ascending=False).to_string(index=False))
print(f'\n🏆 Best LSTM: units={best_lstm["cfg"]["units"]}, embed={best_lstm["cfg"]["embed"]}, lr={best_lstm["cfg"]["lr"]} | F1={best_lstm["f1"]:.4f}')
print(classification_report(yte_l, best_lstm['pred'], target_names=le2.classes_, zero_division=0))

In [ ]:
# Cell 9 - Final Summary
print('='*60)
print('📊 FINAL SUMMARY')
print('='*60)
print(f'🥇 Best Classical ML: {best["model"]} | {best["params"]} | {best["rep"]} | F1={best["f1"]:.4f}')
print(f'🥇 Best FNN:  lr={best_fnn["cfg"]["lr"]}, drop={best_fnn["cfg"]["drop"]}, units={best_fnn["cfg"]["units"]} | F1={best_fnn["f1"]:.4f}')
print(f'🥇 Best LSTM: units={best_lstm["cfg"]["units"]}, embed={best_lstm["cfg"]["embed"]}, lr={best_lstm["cfg"]["lr"]} | F1={best_lstm["f1"]:.4f}')
print('='*60)